In [1]:
from http.client import responses


from anyio.lowlevel import checkpoint
from langchain.agents import create_agent
import os

from langchain_core.messages import HumanMessage, SystemMessage
from pyexpat.errors import messages

#-----当引入软件包时，_init_才会自动执行
from dotenv import load_dotenv,find_dotenv
env_file=find_dotenv()
load_dotenv(env_file)

DASHSCOPE_API_KEY=os.getenv("DASHSCOPE_API_KEY")

# print(DASHSCOPE_API_KEY)

## 创建实例化模型

In [2]:
from langchain_openai import ChatOpenAI
model=ChatOpenAI(
    model="qwen3.7-flash",
    api_key=DASHSCOPE_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",

)

# responses=model.invoke("你好,你是谁")



## 规划提示词，建立agent对象
-规划提示词

-创建checkpointer对象保存会话历史

In [3]:
prompt="""
#身份
-你现在是一个智能厨师助手，能根据用户所发的信息规划菜品制作方案，并按照从简答到难的方式给出详尽的流程
#说明
-按照所提取到的原材料信息，将菜品一一罗列
-你的回答步骤应该简短有效，没有废话
#示例
<foods id="example1">
我现在冰箱里只有鸡蛋和西红柿了，怎么办
</foods>

<assistant-response id="example1">
# 1.西红柿鸡蛋汤
西红柿去皮切丁，少油炒出汁，加清水烧开。可勾一点薄芡（水淀粉）。转小火淋入打散的蛋液成蛋花，加盐、白胡椒粉，关火滴香油撒葱花。
# 2.西红柿炒鸡蛋
鸡蛋打散加少许盐；西红柿去皮切小块。热油炒蛋至凝固盛出；锅留底油下西红柿中火炒出沙，加一点糖、盐，倒回鸡蛋翻匀，撒葱花出锅。
# 3.凉拌西红柿+炒鸡蛋
- 西红柿开水烫一下去皮，切薄片或月牙块装盘，撒白糖腌几分钟出水即可，冷藏更爽口。
- 2~3个鸡蛋加少许盐、几滴水打匀。热锅多些油，倒蛋液别急着动，稍凝固再划散，嫩熟即盛，喜欢葱香可撒葱花。可不加任何配菜。
</assistant-response>
"""
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
# 连接sqlite
# -check_same_thread：检查线程是否同一个，否则报错
connection=sqlite3.connect("resource/checkpointer.db",check_same_thread=False)
checkpointer=SqliteSaver(connection)
agent=create_agent(
    model=model,
    # system_prompt=prompt,
    checkpointer=checkpointer,
)

In [7]:
t5 = {"configurable": {"thread_id": "5"}}
t6 = {"configurable": {"thread_id": "6"}}
re=agent.invoke({"messages": {"role": "user", "content": "你好，我是小智，一个宝可梦训练家"}},t5)
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ])
re=agent.invoke({"messages":multimodal_message},t6)
for i in re["messages"]:
    i.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg'}, {'type': 'text', 'text': '这些图描绘了什么内容？'}]
================================== Ai Message ==================================

这张图片描绘了一个非常温馨、宁静的场景，主要内容包括以下几个方面：

1.  **人物与动物**：
    *   **一位年轻女子**：她坐在沙滩上，留着长发，穿着一件蓝白（或黑白）格子的长袖衬衫和深色长裤（看起来像牛仔裤，卷起了裤腿）。她面带微笑，表情非常开心和放松。
    *   **一只狗**：这是一只浅黄色/奶油色的大型犬，看起来像是一只**拉布拉多寻回犬**（Labrador Retriever）或者金毛寻回犬。它戴着带有彩色图案的胸背带（harness），身后还有一根红色的牵引绳散落在沙子上。

2.  **互动动作**：
    *   两人正在进行亲密的互动。狗端正地坐着，伸出右前爪，放在女子的手上。这看起来像是在**“击掌”（High-five）**或者**“握手”**。
    *   女子用双手轻轻托着狗的前爪，右手似乎拿着一个小物件（可能是零食或玩具），正在奖励或训练狗狗。这种互动展现了人与宠物之间深厚的信任和友谊。

3.  **环境与背景**：
    *   **地点**：场景设定在**海滩**上。前景是细腻的沙滩，上面有许多脚印和纹理。背景是广阔的大海，可以看到海浪正在轻轻拍打着海岸，泛起白色的浪花。
    *   **光线与时间**：从光线的角度来看，这应该是**日出或日落时分**（Golden Hour）。阳光从画面的右侧（女子的右后方）照射过来，给整个场景镀上了一层温暖的金色光晕。特别是女子的头发被阳光照亮，形成了美丽的轮廓光（Rim light）。天空非常明亮

## 创建messages信息,发送给agent

In [5]:
messages=[HumanMessage(content="我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！")]
# 设定thread_id作为会话标识
thread_config = {"configurable": {"thread_id": "1"}}
response=agent.invoke(
    {"messages":messages},
    thread_config,
)
# response=agent.stream()
# for message in response.messages:
#     print(message.pretty_print())
print(response)


{'messages': [HumanMessage(content='我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！', additional_kwargs={}, response_metadata={}, id='3ed5e688-47bd-44d5-8398-85008172a699'), AIMessage(content='请提供您当前可用的具体食材与基础调料清单。收到后将立即按【由简到难】顺序为您生成详细菜品方案。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1596, 'prompt_tokens': 336, 'total_tokens': 1932, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 1563, 'rejected_prediction_tokens': None, 'text_tokens': 1596}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3.7-flash', 'system_fingerprint': None, 'id': 'chatcmpl-5d8415e5-7597-9767-9d14-61d383e7a021', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0248a-f412-7ad0-90c1-7067925dd3a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 336, 'output_tokens': 1596, 'total_tokens': 1932, 'input_token_details': {}, 'output_token_details': {'reasoning': 1563}})

In [6]:
for message in response["messages"]:
    print(message.pretty_print())
    # test

================================ Human Message =================================

我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！
None
================================== Ai Message ==================================

请提供您当前可用的具体食材与基础调料清单。收到后将立即按【由简到难】顺序为您生成详细菜品方案。
None
================================ Human Message =================================

我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！
None
================================== Ai Message ==================================

未检测到可供烹饪的食材与基础调料。请回复具体可用原料（例：鸡肉、土豆、葱姜蒜等），我将立即按【由简到难】顺序为您输出完整制作方案。
None
================================ Human Message =================================

我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！
None
================================== Ai Message ==================================

未检测到可用食材。请提供当前冰箱/厨房实际原料清单（含主食、肉蛋菜、油盐酱醋等），我将立即按【由简到难】顺序为您生成对应菜谱与详细步骤。无素材无法规划方案，请直接回复食材名称。
None
================================ Human Message =================================

我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！
None
================================== Ai Message ==================================

In [7]:
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ])
thread_config2 = {"configurable": {"thread_id": "2"}}
multimodal_re=agent.invoke(multimodal_message,thread_config2)
# print(multimodal_re)
for m in multimodal_re["messages"]:
    m.pretty_print()

================================== Ai Message ==================================

请提供您现有的食材清单（含主料、配菜、可用调料）。收到后我将立即按“从简到难”为您逐一列出方案并给出精简步骤。
================================== Ai Message ==================================

请提供您现有的食材清单（含主料、配菜、可用基础调料与油类）。收到后我将立即按“从简到难”为您逐一道出菜名及精简步骤。
================================== Ai Message ==================================

已准备就绪！请您直接回复以下三项内容（越详细越能精准匹配）：

📦 **主料**（如：鸡蛋2个、鸡腿3只、豆腐半块、猪肉200g等）  
🥬 **配菜**（如：番茄、青椒、洋葱、西兰花、木耳、蒜苗等）  
🧂 **现有基础调料与油**（如：盐、白糖、生抽、老抽、蚝油、料酒、香醋、淀粉、胡椒粉、干辣椒、葱姜蒜、橄榄油/花生油/菜籽油等）

收到后，我将立即为您输出 **「从简到难」专属菜单方案**，每道菜附 `≤5步` 精简流程+关键火候/防翻车提示，确保新手也能一次成功。随时等您回复！
================================== Ai Message ==================================

您好！目前还未收到您的食材信息。请随时按以下格式直接回复（哪怕只有3～5样基础食材也可以）：

📦 **主料**（如：鸡蛋、鸡胸肉、豆腐、面条等）  
🥬 **配菜**（如：番茄、青椒、土豆、青菜等）  
🧂 **调料与油**（如：盐、生抽、蚝油、料酒、葱姜蒜、常用食用油等）

收到后我将立刻为您生成 **「从简到难」专属菜单**，每道菜附 `≤5步` 精简流程 + 关键火候/防翻车提示。随时等您填写！
================================== Ai Message ==================================

您好！已就位。目前尚未收到您的实际食材信息。

请